# SFT with native multi-token prediction

Fine-tune Qwen3.5 and its native MTP head together against assistant responses.
The default trains LoRA adapters on a quantized target together with the full
floating-point MTP head. The original base weights stay frozen.
Set `TRAIN_TYPE = "full"` to train all target weights and the MTP head together.
Set `TRAIN_TYPE = "head"` to train only the predictor with a frozen target instead.
Full fine-tuning requires more memory than a quantized target with head-only training.

The predictor combines `h[t]` with ground-truth `x[t+1]` to predict `x[t+2]`.
Prompt and padding positions are excluded. The objective is next-token SFT plus
weighted next-next-token CE. Held-out loss does not measure speculative acceptance
or guarantee a decoding speedup.


## Setup

Use an Apple Silicon Mac and the Python environment selected by your notebook kernel.
Install once from your checkout, then restart the kernel:

```sh
python -m pip install -e "/absolute/path/to/mlx-vlm[train]" jupyterlab ipykernel
```

Run the cells in order. The first load downloads the model and dataset.


In [ ]:
import json
from dataclasses import asdict
from pathlib import Path

import mlx.core as mx
import mlx.optimizers as optim
import pandas as pd
from datasets import load_dataset
from mlx.utils import tree_flatten

from mlx_vlm import load
from mlx_vlm.prompt_utils import apply_chat_template
from mlx_vlm.speculative.drafters import load_drafter
from mlx_vlm.trainer import Trainer, VLMTrainingArgs, attach_mtp_head, prepare_model_for_training


## 1. Settings

Use the same base/tokenizer family as the head. `MTP_MODEL_ID` can also be a local
floating-point split-head directory. Loading from an original Hub checkpoint may
require downloading its weight shards even though the splitter retains only MTP weights.
`TRAIN_TYPE` selects `"lora"`, `"full"`, or `"head"`. `LORA_RANK`, `LORA_ALPHA`,
and `LORA_DROPOUT` configure new adapters. LoRA and head-only training can use
a quantized target. Full training dequantizes a quantized target;
loading floating-point weights directly avoids that extra conversion.
`CHECKPOINT_PATH` restores the weights saved in the selected mode; optimizer state starts fresh.


In [ ]:
MODEL_ID = "mlx-community/Qwen3.5-2B-4bit"
MTP_MODEL_ID = "Qwen/Qwen3.5-2B"
DATASET_ID = "mlx-community/JOSIE-v2-Instruct-5K"
TRAIN_TYPE = "lora"
LORA_RANK = 8
LORA_ALPHA = 16.0
LORA_DROPOUT = 0.0
CHECKPOINT_PATH = None
OUTPUT_DIR = Path("adapters/vlm/sft_mtp")
EXPORT_DIR = OUTPUT_DIR / "mtp_drafter"
VALIDATION_FRACTION = 0.05
MAX_TRAIN_SAMPLES = 256
MAX_EVAL_SAMPLES = 32
SEED = 42

args = VLMTrainingArgs(
    mtp_loss_weight=0.3,
    iters=100,
    batch_size=1,
    gradient_accumulation_steps=4,
    max_seq_length=1024,
    train_on_completions=True,
    grad_checkpoint=False,
    cache_size=8,
    steps_per_report=10,
    steps_per_eval=50,
    steps_per_save=100,
    adapter_file=str(OUTPUT_DIR / "weights.safetensors"),
)
mx.random.seed(SEED)
args.validate()


## 2. Data

JOSIE already contains `messages`. Remove conversations without a non-empty final
assistant response, then create a seeded training/validation split.


In [ ]:
dataset = load_dataset(DATASET_ID, split="train")
if "messages" not in dataset.column_names:
    raise ValueError(f"Expected a messages column, got {dataset.column_names}.")


def has_assistant_target(row):
    messages = row["messages"]
    return bool(
        messages
        and messages[-1].get("role") == "assistant"
        and isinstance(messages[-1].get("content"), str)
        and messages[-1]["content"].strip()
    )


source_rows = len(dataset)
dataset = dataset.filter(
    has_assistant_target, desc="Keep conversations with assistant targets"
)
dropped_rows = source_rows - len(dataset)
print({"source_rows": source_rows, "dropped_without_assistant_target": dropped_rows})
if len(dataset) < 2:
    raise ValueError("Need at least two conversations with assistant targets to split.")

partitions = dataset.train_test_split(test_size=VALIDATION_FRACTION, seed=SEED)


def limit_rows(rows, maximum):
    if maximum is None:
        return rows
    if maximum < 1:
        raise ValueError("Sample limits must be positive or None.")
    return rows.select(range(min(maximum, len(rows))))


train_rows = limit_rows(partitions["train"], MAX_TRAIN_SAMPLES)
eval_rows = limit_rows(partitions["test"], MAX_EVAL_SAMPLES)

assert len(train_rows) >= args.batch_size, "Reduce batch_size or add training rows."
assert len(eval_rows) >= args.batch_size, "Validation needs a complete batch."
print({"train_rows": len(train_rows), "validation_rows": len(eval_rows)})
print(train_rows.features)

## 3. Prepare the target and attach the native predictor

For LoRA, prepare the base adapters first, then attach the floating-point MTP
head. Only base adapters and the head are trainable.
For full training, attach the head before model preparation so a full checkpoint
restores both target and predictor. Preparation unfreezes the entire model.
In head-only mode, freeze the target first and restore only the predictor weights.
Keep the head unbound until inference so shared base modules are not duplicated.


In [ ]:
def load_training_model(checkpoint_path=None):
    if TRAIN_TYPE not in {"lora", "full", "head"}:
        raise ValueError('TRAIN_TYPE must be "lora", "full", or "head".')
    target, processor = load(MODEL_ID)
    head, kind = load_drafter(MTP_MODEL_ID, kind="mtp")
    assert kind == "mtp"
    if TRAIN_TYPE == "full":
        attach_mtp_head(target, head)
        prepare_model_for_training(
            target, train_type="full", checkpoint_path=checkpoint_path,
        )
    elif TRAIN_TYPE == "lora":
        prepare_model_for_training(
            target, train_type="lora", lora_rank=LORA_RANK,
            lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
            checkpoint_path=checkpoint_path,
        )
        attach_mtp_head(target, head, checkpoint_path=checkpoint_path)
    else:
        target.freeze()
        attach_mtp_head(target, head, checkpoint_path=checkpoint_path)
    return target, processor


model, processor = load_training_model(CHECKPOINT_PATH)
trainable_names = [name for name, _ in tree_flatten(model.trainable_parameters())]
assert any(name.startswith("mtp_head.") for name in trainable_names)
if TRAIN_TYPE == "full":
    assert set(trainable_names) == {name for name, _ in tree_flatten(model.parameters())}
elif TRAIN_TYPE == "lora":
    assert any(name.endswith("lora_b") for name in trainable_names)
    assert all(
        name.startswith("mtp_head.") or name.endswith(("lora_a", "lora_b"))
        for name in trainable_names
    )
else:
    assert all(name.startswith("mtp_head.") for name in trainable_names)
print({"trainable_tensors": len(trainable_names), "train_type": TRAIN_TYPE})
formatted_sample = apply_chat_template(
    processor, model.config, train_rows[0]["messages"],
    num_images=0, add_generation_prompt=False,
)
print(formatted_sample[:2000])


## 4. Train and evaluate

The SFT objective reports base next-token NLL plus weighted MTP CE. LoRA updates the base adapters and the full MTP head together.
Full training
updates the target and MTP head together, including gradients from MTP through
the target hidden states and shared embeddings/output projection. In head-only
mode, base NLL is constant with respect to the trainable parameters.
Monitor both `nll` and `mtp_loss` on held-out responses.


In [ ]:
optimizer = optim.AdamW(learning_rate=1e-4, weight_decay=0.0)
trainer = Trainer(
    model, args, optimizer=optimizer, algorithm="sft", processing_class=processor,
    train_dataset=train_rows, eval_dataset=eval_rows, seed=SEED,
)
baseline = trainer.evaluate()
metrics = trainer.train()
validation = trainer.evaluate()
assert mx.isfinite(mx.array(validation["val_mtp_loss"])).item()
pd.DataFrame([baseline, validation], index=["before", "after"]).reindex(
    columns=["val_mtp_loss", "val_num_mtp_tokens", "val_nll", "val_loss"]
)


## 5. Save and verify a training checkpoint

In LoRA mode, the checkpoint contains base adapters and `mtp_head.*` tensors,
with adapter settings in `adapter_config.json`.
In full mode, `weights.safetensors` contains the target and `mtp_head.*` tensors.
In head-only mode, it contains only `mtp_head.*`. Load a fresh model with the same
preparation order and verify the full objective as well as the MTP loss.
Reloading a second full model also requires additional memory.


In [ ]:
trainer.save_adapter()
saved_weights = mx.load(args.adapter_file)
assert any(name.startswith("mtp_head.") for name in saved_weights)
if TRAIN_TYPE == "full":
    assert any(name.startswith("language_model.") for name in saved_weights)
elif TRAIN_TYPE == "lora":
    assert any(name.endswith("lora_b") for name in saved_weights)
    assert all(
        name.startswith("mtp_head.") or name.endswith(("lora_a", "lora_b"))
        for name in saved_weights
    )
else:
    assert all(name.startswith("mtp_head.") for name in saved_weights)
reloaded_model, reloaded_processor = load_training_model(args.adapter_file)
reloaded_trainer = Trainer(
    reloaded_model, args, algorithm="sft", processing_class=reloaded_processor,
    eval_dataset=eval_rows, seed=SEED,
)
reloaded_validation = reloaded_trainer.evaluate()
for metric in ("val_mtp_loss", "val_nll", "val_loss"):
    assert mx.allclose(
        mx.array(validation[metric]), mx.array(reloaded_validation[metric]),
        rtol=1e-5, atol=1e-5,
    ).item()


## 6. Export a standalone drafter

Export unprefixed native head weights and its configuration. The drafter uses
its target's shared embeddings and output projection at inference. Pair it with
the same base with its trained adapters in LoRA mode, the fine-tuned target
checkpoint in full mode, or the original frozen target in
head-only mode. Exporting the head alone does not export the fine-tuned target.
The tokenizer stays with the target. The final cell reloads the head artifact.


In [ ]:
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
export_config = asdict(model.mtp_head.config)
export_config["text_config"]["mtp_num_hidden_layers"] = len(model.mtp_head.layers)
(EXPORT_DIR / "config.json").write_text(json.dumps(export_config, indent=2) + "\n", encoding="utf-8")
mx.save_safetensors(
    str(EXPORT_DIR / "model.safetensors"),
    dict(tree_flatten(model.mtp_head.parameters())),
)
(EXPORT_DIR / "training_info.json").write_text(
    json.dumps({
        "target_model": MODEL_ID, "target_checkpoint": str(Path(args.adapter_file).resolve()) if TRAIN_TYPE != "head" else None,
        "source_mtp_model": MTP_MODEL_ID, "dataset": DATASET_ID,
        "seed": SEED, "training_args": asdict(args), "head_only": TRAIN_TYPE == "head", "train_type": TRAIN_TYPE,
    }, indent=2) + "\n", encoding="utf-8",
)
exported_head, exported_kind = load_drafter(str(EXPORT_DIR), kind="mtp")
assert exported_kind == "mtp"
for name, value in tree_flatten(exported_head.parameters()):
    assert mx.allclose(value, dict(tree_flatten(model.mtp_head.parameters()))[name]).item()
print(EXPORT_DIR.resolve())


## 7. Use the trained head for speculative decoding

`reloaded_model` contains the restored target and `exported_head` contains the
standalone predictor. For LoRA, restore the base adapters alongside the drafter.
For full training, use the restored target with its updated shared weights.
In a fresh session, restore it with `load_training_model(args.adapter_file)`
before passing it and the exported drafter to speculative generation.

Compare generated quality, acceptance rate, memory, and throughput on held-out
prompts. See the [training guide](../../docs/training.md) for checkpoint handling.
